# Travel Guide Knowledge Worker

A conversational AI that answers questions from a personal travel knowledge base using Advanced RAG.

**Pipeline:**
1. **Knowledge Base** - 10 markdown files covering destinations, tips, and itineraries
2. **LLM-Based Chunking** - Intelligent document splitting with headlines and summaries
3. **Vector Storage** - ChromaDB with OpenAI embeddings
4. **Advanced RAG** - Query rewriting + dual retrieval + LLM reranking
5. **Gradio Chat UI** - Two-column layout with chat and context display

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
from tenacity import retry, wait_exponential
import gradio as gr

load_dotenv(override=True)

MODEL = "openai/gpt-4.1-nano"
DB_NAME = "travel_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
AVERAGE_CHUNK_SIZE = 100
RETRIEVAL_K = 20
FINAL_K = 10
KNOWLEDGE_BASE_PATH = Path("travel-knowledge-base")
wait = wait_exponential(multiplier=1, min=10, max=240)

openai = OpenAI(base_url="https://openrouter.ai/api/v1")
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)


def llm_call(messages, **kwargs):
    return completion(base_url="https://openrouter.ai/api/v1", model=MODEL, messages=messages, **kwargs)

## Section 2: Knowledge Base (Generated On-the-Fly)

The knowledge base consists of 10 markdown files organized into three categories, generated at runtime via LLM:
- **Destinations** (4 files): Tokyo, Paris, Cape Town, Cusco
- **Tips** (3 files): Budget travel, packing essentials, safety abroad
- **Itineraries** (3 files): Southeast Asia 2 weeks, European highlights 10 days, East Africa safari

In [ ]:
KB_FILES = {
    "destinations": ["tokyo", "paris", "cape-town", "cusco"],
    "tips": ["budget-travel", "packing-essentials", "safety-abroad"],
    "itineraries": ["southeast-asia-2-weeks", "european-highlights-10-days", "east-africa-safari"],
}

GENERATE_PROMPT = """You are a travel content writer creating a detailed markdown guide.

Write a comprehensive travel guide document in markdown for: {topic}
Category: {category}

The document should be approximately 3000-5000 characters and include these sections with ## headings:
{sections}

Use bullet points with **bold** labels for specific recommendations.
Include practical, actionable advice with specific names of places, restaurants, and attractions.
Start with a # heading using the proper name of the destination/topic.
"""

SECTION_MAP = {
    "destinations": "Overview, Best Time to Visit, Top Attractions (5-7 items), Food & Dining (4-5 items), Getting Around, Budget Tips (5 items), Cultural Etiquette (5 items)",
    "tips": "Overview, Key Principles (5-6 items), Practical Tips (5-6 items), Common Mistakes to Avoid, Pro Tips, Resources & Tools",
    "itineraries": "Overview, Best Time to Go, Day-by-Day Itinerary (detailed), Budget Breakdown, Packing List, Important Tips",
}


def generate_knowledge_base():
    if KNOWLEDGE_BASE_PATH.exists() and any(KNOWLEDGE_BASE_PATH.rglob("*.md")):
        print("Knowledge base already exists, skipping generation.")
        return

    print("Generating knowledge base files...")
    for category, topics in KB_FILES.items():
        folder = KNOWLEDGE_BASE_PATH / category
        folder.mkdir(parents=True, exist_ok=True)
        for topic in topics:
            filepath = folder / f"{topic}.md"
            display_topic = topic.replace("-", " ").title()
            prompt = GENERATE_PROMPT.format(
                topic=display_topic, category=category, sections=SECTION_MAP[category]
            )
            response = llm_call([{"role": "user", "content": prompt}])
            content = response.choices[0].message.content
            filepath.write_text(content, encoding="utf-8")
            print(f"  Created {filepath} ({len(content):,} chars)")

    print(f"Generated {sum(len(v) for v in KB_FILES.values())} files.")


generate_knowledge_base()

In [2]:
for folder in sorted(KNOWLEDGE_BASE_PATH.iterdir()):
    if folder.is_dir():
        files = sorted(folder.glob("*.md"))
        print(f"\n{folder.name}/ ({len(files)} files)")
        for f in files:
            print(f"  {f.name} ({f.stat().st_size:,} bytes)")


destinations/ (4 files)
  cape-town.md (4,070 bytes)
  cusco.md (4,231 bytes)
  paris.md (3,839 bytes)
  tokyo.md (3,328 bytes)

itineraries/ (3 files)
  east-africa-safari.md (6,635 bytes)
  european-highlights-10-days.md (5,460 bytes)
  southeast-asia-2-weeks.md (4,556 bytes)

tips/ (3 files)
  budget-travel.md (4,081 bytes)
  packing-essentials.md (3,814 bytes)
  safety-abroad.md (4,698 bytes)


## Section 3: Ingestion & Vectorization

We use LLM-based chunking: instead of splitting on character counts, we ask GPT to intelligently split each document into overlapping chunks with headlines and summaries. This produces higher quality retrieval results.

In [3]:
class Result(BaseModel):
    page_content: str
    metadata: dict


class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query",
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
        )


class Chunks(BaseModel):
    chunks: list[Chunk]

In [4]:
def fetch_documents():
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

documents = fetch_documents()

Loaded 10 documents


In [6]:
documents[3]

{'type': 'destinations',
 'source': 'travel-knowledge-base/destinations/paris.md',
 'text': '# Paris, France\n\n## Overview\nParis, the City of Light, is the capital of France and one of the most visited cities in the world. Known for its art, fashion, gastronomy, and culture, Paris is a city that captivates visitors with its elegant boulevards, world-renowned museums, and romantic atmosphere. The city is divided into 20 arrondissements (districts), each with its own distinct character.\n\n## Best Time to Visit\nThe best time to visit Paris is from April to June and September to October, when the weather is pleasant and crowds are manageable. Summer (July-August) is peak tourist season with warm weather but many locals leave for vacation. Winter (November-February) is cold and grey but offers lower prices, holiday markets, and fewer tourists.\n\n## Top Attractions\n- **Eiffel Tower**: The iconic iron lattice tower offers breathtaking views from three levels. Book tickets online to skip

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from a travel guide knowledge base.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about travel destinations, tips, and itineraries.
You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""


def make_messages(document):
    return [{"role": "user", "content": make_prompt(document)}]


@retry(wait=wait)
def process_document(document):
    messages = make_messages(document)
    response = llm_call(messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [8]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents, desc="Chunking documents"):
        result = process_document(doc)
        chunks.extend(result)
    return chunks

chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")

Chunking documents:   0%|          | 0/10 [00:00<?, ?it/s]

Chunking documents: 100%|██████████| 10/10 [02:58<00:00, 17.88s/it]

Created 71 chunks from 10 documents


In [9]:
def create_embeddings(chunks):
    global collection
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

create_embeddings(chunks)

Vectorstore created with 71 documents


## Section 4: Advanced RAG Pipeline

The RAG pipeline uses three techniques for high-quality retrieval:
1. **Query Rewriting** - Rewrites the user's question for better knowledge base matching
2. **Dual Retrieval** - Retrieves chunks using both the original and rewritten queries, then merges
3. **LLM Reranking** - Reranks merged chunks by relevance using an LLM

In [10]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


@retry(wait=wait)
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = llm_call(messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    return [chunks[i - 1] for i in order]

In [11]:
@retry(wait=wait)
def rewrite_query(question, history=None):
    history = history or []
    message = f"""
You are in a conversation with a user, answering questions about travel destinations, tips, and itineraries.
You are about to look up information in a Travel Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a short, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
"""
    response = llm_call([{"role": "system", "content": message}])
    return response.choices[0].message.content

In [12]:
def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


def merge_chunks(chunks, reranked):
    merged = chunks[:]
    existing = {chunk.page_content for chunk in chunks}
    for chunk in reranked:
        if chunk.page_content not in existing:
            merged.append(chunk)
            existing.add(chunk.page_content)
    return merged


def fetch_context(original_question):
    rewritten_question = rewrite_query(original_question)
    chunks1 = fetch_context_unranked(original_question)
    chunks2 = fetch_context_unranked(rewritten_question)
    chunks = merge_chunks(chunks1, chunks2)
    reranked = rerank(original_question, chunks)
    return reranked[:FINAL_K]

In [13]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly travel guide assistant.
You help users plan trips, learn about destinations, and get practical travel advice.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Travel Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""


def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )


@retry(wait=wait)
def answer_question(question: str, history: list[dict] | None = None) -> tuple[str, list]:
    history = history or []
    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = llm_call(messages)
    return response.choices[0].message.content, chunks

In [14]:
answer, context = answer_question("What are the best things to do in Tokyo?")
print(answer)
print(f"\n--- Retrieved {len(context)} chunks ---")
for c in context[:3]:
    print(f"  Source: {c.metadata['source']}")

The best things to do in Tokyo include a mix of cultural, historic, shopping, and entertainment experiences:

1. Visit **Senso-ji Temple** in Asakusa, Tokyo’s oldest temple, and explore Nakamise shopping street for souvenirs and street food.
2. Experience the bustling **Shibuya Crossing**, the world's busiest pedestrian scramble, with great views from nearby cafes or the Shibuya Sky observation deck.
3. Enjoy the peaceful atmosphere at **Meiji Shrine**, nestled within a forested park near Harajuku.
4. Explore **Tsukiji Outer Market** for fresh sushi and street food, even after the inner market moved to Toyosu.
5. Dive into anime, manga, and electronics culture in **Akihabara**.
6. Relax in **Shinjuku Gyoen**, one of Tokyo's largest parks, perfect for cherry blossom viewing in spring.
7. Take in panoramic city views from **Tokyo Skytree**, which offers views on clear days of Mount Fuji.
8. Indulge in Tokyo’s renowned culinary scene, from high-end sushi and tempura restaurants to afforda

## Section 5: Gradio Chat UI

A two-column layout with chat on the left and retrieved context on the right.

In [15]:
def format_context(context):
    result = "<h2 style='color: #2e86ab;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #2e86ab;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result


def chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)


def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Travel Guide Assistant", theme=theme) as ui:
    gr.Markdown("# Travel Guide Assistant\nAsk me anything about travel destinations, tips, and itineraries!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about travel...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Example Questions to Try

- "What's the best time to visit Tokyo?"
- "Budget tips for traveling in Europe?"
- "Plan a 2-week Southeast Asia trip"
- "What should I pack for a safari?"
- "How do I prevent altitude sickness in Cusco?"
- "What are the must-try foods in Paris?"
- "Is Cape Town safe for tourists?"
- "How much does an East Africa safari cost?"